# Project GEO876 - Wildfire Mapping
Janis Büchel

### Load the API-url

In [41]:
# Install Packages:

import pandas as pd
import numpy as np
import requests
from datetime import datetime



#### Map Key and API-url

To access the API, a map key is required, which can be requested on the website.

In [42]:
#Set map key:
MAP_KEY = "e5b967caf968ec1283cb213a30e9b9d4"

# Set base url:
url = "https://firms.modaps.eosdis.nasa.gov/api/"



First step: shwo which sensors are available on the API:

In [43]:
# Create the correct API-url with the map key and show the list of sensors.
data_availability_url = f"{url}data_availability/csv/{MAP_KEY}/all"
df_data_avialiability = pd.read_csv(data_availability_url)
print("Following sensors are available:")
display(df_data_avialiability)



Following sensors are available:


,data_id,min_date,max_date
0,MODIS_NRT,2026-01-01,2026-05-03
1,MODIS_SP,2000-11-01,2025-12-31
2,VIIRS_NOAA20_NRT,2026-03-01,2026-05-03
3,VIIRS_NOAA20_SP,2018-04-01,2026-02-28
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-03
5,VIIRS_SNPP_NRT,2026-03-01,2026-05-03
6,VIIRS_SNPP_SP,2012-01-20,2026-02-28
7,LANDSAT_NRT,2022-06-20,2026-05-02
8,GOES_NRT,2022-08-09,2026-05-03
9,BA_MODIS,2000-11-01,2026-02-01


Filter for sensors which recorded wildfired today

In [44]:
# Convert max_date into a date
df_data_avialiability["max_date"] = pd.to_datetime(df_data_avialiability['max_date'])

# Date of today
date_today = pd.Timestamp.now().normalize()
print(date_today)

# Filter for sensors with recordings today
df_available_sens = df_data_avialiability[df_data_avialiability["max_date"] == date_today]
print(df_available_sens)

# Save data_id in a list
data_id = df_available_sens["data_id"].tolist()
print(data_id)

2026-05-03 00:00:00
            data_id    min_date   max_date
0         MODIS_NRT  2026-01-01 2026-05-03
2  VIIRS_NOAA20_NRT  2026-03-01 2026-05-03
4  VIIRS_NOAA21_NRT  2024-01-17 2026-05-03
5    VIIRS_SNPP_NRT  2026-03-01 2026-05-03
8          GOES_NRT  2022-08-09 2026-05-03
['MODIS_NRT', 'VIIRS_NOAA20_NRT', 'VIIRS_NOAA21_NRT', 'VIIRS_SNPP_NRT', 'GOES_NRT']


Collect data for one sensor (MODIS):

In [45]:
# Shows the data of the sensor for the desired area (world = [-180,-90,180,90]) and for the day range.
sensor = "MODIS_NRT"
focus_area = "world"
day_range = 5 # max. 5 days

area_url = f"{url}area/csv/{MAP_KEY}/{sensor}/{focus_area}/{day_range}"
df_area = pd.read_csv(area_url)

df_area

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,31.04159,49.40215,302.75,1.51,1.21,2026-04-29,3,Aqua,MODIS,51,6.1NRT,291.98,9.53,N
1,31.04328,49.38674,302.18,1.52,1.22,2026-04-29,3,Aqua,MODIS,47,6.1NRT,291.72,8.90,N
2,31.12479,49.33591,312.84,1.53,1.22,2026-04-29,3,Aqua,MODIS,86,6.1NRT,291.78,21.46,N
3,31.12657,49.32035,328.94,1.54,1.22,2026-04-29,3,Aqua,MODIS,100,6.1NRT,292.87,51.52,N
4,31.13726,49.32189,320.92,1.54,1.22,2026-04-29,3,Aqua,MODIS,100,6.1NRT,293.01,35.16,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27443,13.27215,-5.34524,315.90,1.06,1.03,2026-05-03,917,Terra,MODIS,37,6.1NRT,285.76,8.18,D
27444,13.27345,-5.35478,313.35,1.06,1.03,2026-05-03,917,Terra,MODIS,23,6.1NRT,287.13,5.73,D
27445,13.38619,-7.70340,324.28,1.42,1.18,2026-05-03,917,Terra,MODIS,62,6.1NRT,305.52,9.96,D
27446,14.12366,-10.41263,338.74,2.34,1.48,2026-05-03,917,Terra,MODIS,86,6.1NRT,302.64,98.37,D


Collect data for 3 VIIRS sensors:

In [46]:
# Shows the data of the sensor for the desired area (world = [-180,-90,180,90]) and for the day range.
sensors = ["VIIRS_NOAA20_NRT", "VIIRS_NOAA21_NRT", "VIIRS_SNPP_NRT"]
focus_area = "world"
day_range = 1 # max. 5 days

# df for with data from all three sensors:
df_viirs = []

for foc_sensor in sensors:
    area_url = f"{url}area/csv/{MAP_KEY}/{foc_sensor}/{focus_area}/{day_range}"
    df_sens = pd.read_csv(area_url)
    df_sens["sensor_id"] = foc_sensor
    df_viirs.append(df_sens)
    print(f"Data for {foc_sensor} loaded, total {len(df_sens)} rows.")

# Add all three dataframes into one big dataframe
df_viirs = pd.concat(df_viirs, ignore_index=True)
print(f"Total number of rows: {len(df_viirs)}\nFinal df:")
print(df_viirs.tail(5))

Data for VIIRS_NOAA20_NRT loaded, total 9429 rows.
Data for VIIRS_NOAA21_NRT loaded, total 11875 rows.
Data for VIIRS_SNPP_NRT loaded, total 6793 rows.
Total number of rows: 28097
Final df:
       latitude  longitude  bright_ti4  scan  track    acq_date  acq_time  \
28092  33.35072 -112.60801      327.35   0.4    0.4  2026-05-03       934   
28093  33.95387 -117.12125      299.65   0.4    0.4  2026-05-03       934   
28094  34.14204 -117.42616      306.75   0.4    0.4  2026-05-03       934   
28095  34.60838 -117.33710      313.96   0.4    0.4  2026-05-03       934   
28096  34.62341 -117.09969      301.23   0.4    0.4  2026-05-03       934   

      satellite instrument confidence version  bright_ti5   frp daynight  \
28092         N      VIIRS          n  2.0URT      261.15  2.92        N   
28093         N      VIIRS          n  2.0URT      284.01  0.54        N   
28094         N      VIIRS          n  2.0URT      285.40  0.73        N   
28095         N      VIIRS          n  2.0U

Change acq_date und acq_time to datetime:

In [48]:
df_viirs["acq_time"] = df_viirs["acq_time"].astype(str) # Convert time to string
df_viirs["acq_time"] = df_viirs["acq_time"].str.zfill(4) # Fill with zeros

# Create new column with datetime
df_viirs["acq_datetime"] = pd.to_datetime(
    df_viirs["acq_date"] + " " + df_viirs["acq_time"], 
    format='%Y-%m-%d %H%M'
)
# Print new column
print(df_viirs[["acq_datetime"]].tail())

             acq_datetime
28092 2026-05-03 09:34:00
28093 2026-05-03 09:34:00
28094 2026-05-03 09:34:00
28095 2026-05-03 09:34:00
28096 2026-05-03 09:34:00
